In [13]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
np.random.seed(42)


TARGET_VAR = "ActivePower"
FORECAST_HOURS = 15 * 24  
DATA_PATH = "train.csv"
OUT_HOURLY = "predictions_xgboost_15days_hourly.csv"
OUT_DAILY = "predictions_xgboost_15days_daily_totals.csv"
MAX_TRAIN_HOURS = 180 * 24  

TEST_HOURS = FORECAST_HOURS  

N_CV_FOLDS = 3              
CV_VAL_HOURS = TEST_HOURS   
MIN_TRAIN_HOURS = 500       
LAGS = [1, 2, 3, 24, 48, 168]  
ROLLING_WINDOWS = [24, 168]    

N_ESTIMATORS = 150
MAX_DEPTH = 5
LEARNING_RATE = 0.05
SUBSAMPLE = 0.8
COL_SAMPLE_BYTREE = 0.8
N_JOBS = -1
EARLY_STOPPING_ROUNDS = 15

In [14]:
def load_and_resample(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Datetime"], index_col="Datetime")
    df = df.sort_index()
    df = df.resample("10T").mean()
    return df


def impute(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    out = df[numeric_cols].copy()
    for col in out.columns:
        out[col] = out[col].interpolate(method="linear", limit_direction="both")
    out = out.ffill().bfill()
    return out


def hourly_aggregate(df: pd.DataFrame) -> pd.Series:
    return df[TARGET_VAR].resample("H").sum()

In [15]:
def build_features(series: pd.Series, lags: list, windows: list) -> pd.DataFrame:
    out = pd.DataFrame(index=series.index)
    for lag in lags:
        out[f"lag_{lag}"] = series.shift(lag)
    for w in windows:
        out[f"roll_mean_{w}"] = series.shift(1).rolling(window=w, min_periods=1).mean()
    out["hour"] = out.index.hour
    out["day_of_week"] = out.index.dayofweek
    out["target"] = series
    return out.dropna(how="all")

In [ ]:
import xgboost as xgb

df = load_and_resample(DATA_PATH)
print(f"   Shape after 10T resample: {df.shape}")

df_imp = impute(df)

hourly = hourly_aggregate(df_imp)
hourly = hourly.dropna()
print(f"   Hourly series length: {len(hourly)} hours ({len(hourly)/24:.1f} days)")

max_lag = max(LAGS)
feats = build_features(hourly, LAGS, ROLLING_WINDOWS)
feats = feats.dropna(subset=[c for c in feats.columns if c != "target"])
feats = feats.dropna(subset=["target"])
feature_cols = [c for c in feats.columns if c != "target"]
X_full = feats[feature_cols]
y_full = feats["target"]

X_test = X_full.iloc[-TEST_HOURS:]
y_test = y_full.loc[X_test.index]
X_train_pool = X_full.iloc[:-TEST_HOURS]
y_train_pool = y_full.loc[X_train_pool.index]

if MAX_TRAIN_HOURS is not None and len(X_train_pool) > MAX_TRAIN_HOURS:
    X_train_pool = X_train_pool.iloc[-MAX_TRAIN_HOURS:]
    y_train_pool = y_train_pool.reindex(X_train_pool.index).dropna()
    X_train_pool = X_train_pool.loc[y_train_pool.index]
    print(f"   Using last {MAX_TRAIN_HOURS} hours for training ({len(X_train_pool)} rows)")
print(f"   Train: {len(X_train_pool)} hours, Test: {len(X_test)} hours")

X_train, y_train = X_train_pool, y_train_pool
print(f"   Feature matrix (train): {X_train.shape}")

cv_rmse_hourly = []
cv_rmse_daily = []
n_pool = len(X_train_pool)
for fold in range(N_CV_FOLDS):
    val_end = n_pool - fold * CV_VAL_HOURS
    val_start = val_end - CV_VAL_HOURS
    if val_start < MIN_TRAIN_HOURS or val_end <= val_start:
        break
    X_tr = X_train_pool.iloc[:val_start]
    y_tr = y_train_pool.iloc[:val_start]
    X_val = X_train_pool.iloc[val_start:val_end]
    y_val = y_train_pool.iloc[val_start:val_end]
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    params_cv = {
        "objective": "reg:squarederror",
        "max_depth": MAX_DEPTH,
        "learning_rate": LEARNING_RATE,
        "subsample": SUBSAMPLE,
        "colsample_bytree": COL_SAMPLE_BYTREE,
        "n_jobs": N_JOBS,
        "seed": 42,
    }
    m = xgb.train(
        params_cv,
        dtr,
        num_boost_round=N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        evals=[(xgb.DMatrix(X_val, label=y_val), "val")],
        verbose_eval=False,
    )
    pred_val = np.maximum(m.predict(xgb.DMatrix(X_val)), 0.0)
    rmse_h = np.sqrt(np.mean((y_val.values - pred_val) ** 2))
    cv_rmse_hourly.append(rmse_h)
    y_val_s = pd.Series(y_val.values, index=y_val.index)
    p_val_s = pd.Series(pred_val, index=y_val.index)
    ad = y_val_s.resample("D").sum()
    pd_ = p_val_s.resample("D").sum()
    common = ad.index.intersection(pd_.index)
    if len(common) > 0:
        rmse_d = np.sqrt(np.mean((ad.loc[common].values - pd_.loc[common].values) ** 2))
        cv_rmse_daily.append(rmse_d)
    print(f"   Fold {fold + 1}: train size {len(X_tr)}, val size {len(X_val)}, hourly RMSE = {rmse_h:.2f}")
if cv_rmse_hourly:
    print(f"   CV hourly RMSE: {np.mean(cv_rmse_hourly):.2f} ± {np.std(cv_rmse_hourly):.2f} ({len(cv_rmse_hourly)} folds)")
    if cv_rmse_daily:
        print(f"   CV daily RMSE:  {np.mean(cv_rmse_daily):.2f} ± {np.std(cv_rmse_daily):.2f} ({len(cv_rmse_daily)} folds)")

dtrain = xgb.DMatrix(X_train, label=y_train)
params = {
    "objective": "reg:squarederror",
    "max_depth": MAX_DEPTH,
    "learning_rate": LEARNING_RATE,
    "subsample": SUBSAMPLE,
    "colsample_bytree": COL_SAMPLE_BYTREE,
    "n_jobs": N_JOBS,
    "seed": 42,
}
evals = [(dtrain, "train")]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=N_ESTIMATORS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=False,
)
n_rounds = getattr(model, "best_iteration", N_ESTIMATORS) or N_ESTIMATORS
print(f"   Boosting rounds used: {n_rounds}")

y_test_pred = model.predict(xgb.DMatrix(X_test))
y_test_pred = np.maximum(y_test_pred, 0.0)
test_rmse_hourly = np.sqrt(np.mean((y_test.values - y_test_pred) ** 2))
test_mae_hourly = np.mean(np.abs(y_test.values - y_test_pred))
print(f"   Hourly (one-step) RMSE: {test_rmse_hourly:.2f}")
print(f"   Hourly (one-step) MAE:  {test_mae_hourly:.2f}")
y_test_series = pd.Series(y_test.values, index=y_test.index)
pred_test_series = pd.Series(y_test_pred, index=y_test.index)
actual_daily = y_test_series.resample("D").sum()
pred_daily = pred_test_series.resample("D").sum()
common = actual_daily.index.intersection(pred_daily.index)
if len(common) > 0:
    test_rmse_daily = np.sqrt(np.mean((actual_daily.loc[common].values - pred_daily.loc[common].values) ** 2))
    test_mae_daily = np.mean(np.abs(actual_daily.loc[common].values - pred_daily.loc[common].values))
    print(f"   Daily totals RMSE: {test_rmse_daily:.2f}")
    print(f"   Daily totals MAE:  {test_mae_daily:.2f}")

last_ts = hourly.index[-1]
history_len = max(max(LAGS), max(ROLLING_WINDOWS))
history = hourly.values[-history_len:].tolist()
preds = []
for step in range(FORECAST_HOURS):
    next_ts = last_ts + pd.Timedelta(hours=step + 1)
    row = {}
    for lag in LAGS:
        if len(history) >= lag:
            row[f"lag_{lag}"] = history[-lag]
        else:
            row[f"lag_{lag}"] = history[0] if history else np.nan
    for w in ROLLING_WINDOWS:
        if len(history) >= w:
            row[f"roll_mean_{w}"] = np.mean(history[-w:])
        else:
            row[f"roll_mean_{w}"] = np.mean(history) if history else np.nan
    row["hour"] = next_ts.hour
    row["day_of_week"] = next_ts.dayofweek
    X_step = pd.DataFrame([row])[feature_cols]
    pred = model.predict(xgb.DMatrix(X_step))[0]
    pred = max(0.0, float(pred)) 
    preds.append(pred)
    history.append(pred)

pred_index = pd.date_range(last_ts + pd.Timedelta(hours=1), periods=FORECAST_HOURS, freq="H")
pred_series = pd.Series(preds, index=pred_index, name=TARGET_VAR)

hourly_path = Path(OUT_HOURLY)
daily_path = Path(OUT_DAILY)

pred_series.to_csv(hourly_path, index=True, index_label="Datetime")
print(f"   Hourly predictions saved: {hourly_path}")

daily_totals = pred_series.resample("D").sum()
daily_totals.name = "predicted_total_power"
daily_totals.to_csv(daily_path, index=True, index_label="Date")


   Shape after 10T resample: (115882, 19)
   Hourly series length: 19314 hours (804.8 days)
   Using last 4320 hours for training (4320 rows)
   Train: 4320 hours, Test: 360 hours
   Feature matrix (train): (4320, 10)
   Fold 1: train size 3960, val size 360, hourly RMSE = 1195.48
   Fold 2: train size 3600, val size 360, hourly RMSE = 1248.18
   Fold 3: train size 3240, val size 360, hourly RMSE = 1180.32
   CV hourly RMSE: 1207.99 ± 29.08 (3 folds)
   CV daily RMSE:  7236.28 ± 1643.25 (3 folds)
   Boosting rounds used: 149
   Hourly (one-step) RMSE: 1127.83
   Hourly (one-step) MAE:  734.01
   Daily totals RMSE: 3896.64
   Daily totals MAE:  3679.26
   Hourly predictions saved: predictions_xgboost_15days_hourly.csv
